# Who is about to cancel?

The customers' features, read as the analyst role -- email addresses arrive masked, as that role is served them -- and a model trained on them, here in the browser. On a desktop Jupyter the same code reads large pulls over Arrow Flight.

In [ ]:
%pip install provisa-client
import provisa_client

provisa_client.sign_in()

In [ ]:
import pandas as pd

df = provisa_client.query(
    "SELECT f.customer_id, f.email, f.region, f.plan, f.tenure_months, f.logins_per_month, "
    'f.tickets, f.churned FROM "churn"."view_churn_features" AS f',
    role="analyst",
)
numeric = ["tenure_months", "logins_per_month", "tickets", "churned"]
df[numeric] = df[numeric].astype(float)
print(len(df), "customers;", int(df["churned"].sum()), "have cancelled")
df.head()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

X = pd.get_dummies(
    df[["plan", "region", "tenure_months", "logins_per_month", "tickets"]], dtype=float
)
y = df["churned"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7, stratify=y
)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"AUC on customers the model never saw: {auc:.2f}")

In [ ]:
at_risk = df[df["churned"] == 0].assign(risk=model.predict_proba(X[df["churned"] == 0])[:, 1])
at_risk.sort_values("risk", ascending=False)[
    ["customer_id", "email", "plan", "tickets", "risk"]
].head(10)